# Phase 1 — Data Sanitation and Preprocessing
**ITS 2122 — Python for Data Science & AI | Sem 3 2026**  
**Project:** Credit Risk & Customer Financial Health Analytics  
**Dataset:** Default of Credit Card Clients (UCI, N = 30 000)  

---

## Objectives

This phase transforms the raw CSV into a clean, analysis-ready DataFrame.  
Steps followed:

1. Load data and validate shape against UCI documentation  
2. Initial assessment — types, nulls, descriptive statistics  
3. Rename columns to Python-friendly `snake_case`  
4. Decode categorical codes to readable labels  
5. Remove duplicates and validate numeric ranges  
6. Flag known edge cases (delay–no-bill, over-limit, credit balance) 
7. Engineer behavioural features for Phase 3 segmentation  
8. Export the cleaned dataset as `cleaned_credit_data.csv`  


## Imports and Display Settings

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── Display preferences ──────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")
print("Libraries loaded ✓")


Libraries loaded ✓


## Constants

Centralising all magic values (file paths, encode maps) so that they are easy to update and review.

In [5]:
# ── File paths ───────────────────────────────────────────────────────
DATA_PATH = "default of credit card clients.csv"
OUTPUT_PATH = "cleaned_credit_data.csv"

# ── Column rename map ────────────────────────────────────────────────
# Converts original UCI names → snake_case (PEP 8 §3.3)
COLUMN_RENAME_MAP = {
    "ID":                          "customer_id",
    "LIMIT_BAL":                   "credit_limit",
    "SEX":                         "sex",
    "EDUCATION":                   "education",
    "MARRIAGE":                    "marriage",
    "AGE":                         "age",
    "PAY_1":                       "pay_status_1",
    "PAY_2":                       "pay_status_2",
    "PAY_3":                       "pay_status_3",
    "PAY_4":                       "pay_status_4",
    "PAY_5":                       "pay_status_5",
    "PAY_6":                       "pay_status_6",
    "BILL_AMT1":                   "bill_amount_1",
    "BILL_AMT2":                   "bill_amount_2",
    "BILL_AMT3":                   "bill_amount_3",
    "BILL_AMT4":                   "bill_amount_4",
    "BILL_AMT5":                   "bill_amount_5",
    "BILL_AMT6":                   "bill_amount_6",
    "PAY_AMT1":                    "pay_amount_1",
    "PAY_AMT2":                    "pay_amount_2",
    "PAY_AMT3":                    "pay_amount_3",
    "PAY_AMT4":                    "pay_amount_4",
    "PAY_AMT5":                    "pay_amount_5",
    "PAY_AMT6":                    "pay_amount_6",
    "default payment next month":  "default_next_month",
}

# ── SEX decode map ───────────────────────────────────────────────────
SEX_MAP = {1: 'Male', 2: 'Female'}

# ── EDUCATION decode map ─────────────────────────────────────────────
# UCI documents codes 1–4 only.
# Codes 0, 5, 6 are undocumented (345 rows total).
# Decision: reclassified as 'Unknown' rather than dropped,
# preserving all records while avoiding distortion of known groups.
EDUCATION_MAP = {
    1: 'Graduate School',
    2: 'University',
    3: 'High School',
    4: 'Other',
    0: 'Unknown',
    5: 'Unknown',
    6: 'Unknown',
}

# ── MARRIAGE decode map ───────────────────────────────────────────────
# UCI documents codes 1–3 only.
# Code 0 is undocumented (54 rows).
# Decision: reclassified as 'Unknown' — same rationale as EDUCATION.
MARRIAGE_MAP = {
    1: 'Married',
    2: 'Single',
    3: 'Other',
    0: 'Unknown',
}

# ── Column group shortcuts ────────────────────────────────────────────
# Defined after rename; referenced throughout feature engineering.
PAY_STATUS_COLS = [f'pay_status_{i}' for i in range(1, 7)]
BILL_AMOUNT_COLS = [f'bill_amount_{i}' for i in range(1, 7)]
PAY_AMOUNT_COLS = [f'pay_amount_{i}' for i in range(1, 7)]
print("Constants defined ✓")


Constants defined ✓


---
## Step 1 — Load Data

The raw file has **two header rows**:

- Row 0: generic UCI labels (`X1, X2 … X24, Y`)
- Row 1: actual field names (`ID, LIMIT_BAL, SEX …`)

Using `header=1` skips the generic row and reads the correct header.  
We then assert shape against the UCI documentation (30 000 rows × 25 columns).

In [6]:
df = pd.read_csv(DATA_PATH, header=1)

# ── Shape assertion ──────────────────────────────────────────────────
EXPECTED_ROWS, EXPECTED_COLS = 30_000, 25
assert df.shape == (EXPECTED_ROWS, EXPECTED_COLS), (
    f"Unexpected shape {df.shape}; "
    f"expected ({EXPECTED_ROWS}, {EXPECTED_COLS})"
)

print("Dataset loaded successfully.")
print(f"  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]}")
print(f"\nFirst 3 rows:")
df.head(3)


Dataset loaded successfully.
  Rows    : 30,000
  Columns : 25

First 3 rows:


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0


---
## Step 2 — Initial Assessment

A systematic audit using `.info()`, `.describe()`, `.isnull().sum()`, and `.value_counts()` on categorical fields.

### 2a — Data Types and Memory

In [7]:
# Shows column dtypes, non-null counts, and memory footprint.
df.info(memory_usage="deep")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_1                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

### 2b — Descriptive Statistics

`include='all'` covers all numeric columns. `.T` transposes for readability.

In [8]:
df.describe(include="all").T


,count,mean,std,min,25%,50%,75%,max
ID,"30,000.00","15,000.50","8,660.40",1.00,"7,500.75","15,000.50","22,500.25","30,000.00"
LIMIT_BAL,"30,000.00","167,484.32","129,747.66","10,000.00","50,000.00","140,000.00","240,000.00","1,000,000.00"
SEX,"30,000.00",1.60,0.49,1.00,1.00,2.00,2.00,2.00
EDUCATION,"30,000.00",1.85,0.79,0.00,1.00,2.00,2.00,6.00
MARRIAGE,"30,000.00",1.55,0.52,0.00,1.00,2.00,2.00,3.00
AGE,"30,000.00",35.49,9.22,21.00,28.00,34.00,41.00,79.00
PAY_1,"30,000.00",-0.02,1.12,-2.00,-1.00,0.00,0.00,8.00
PAY_2,"30,000.00",-0.13,1.20,-2.00,-1.00,0.00,0.00,8.00
PAY_3,"30,000.00",-0.17,1.20,-2.00,-1.00,0.00,0.00,8.00
PAY_4,"30,000.00",-0.22,1.17,-2.00,-1.00,0.00,0.00,8.00


### 2c — Missing Values

In [9]:
null_counts = df.isnull().sum()
has_nulls = null_counts[null_counts > 0]

if has_nulls.empty:
    print("No missing values found across all 25 columns. ✓")
else:
    print("Columns with missing values:")
    print(has_nulls)


No missing values found across all 25 columns. ✓


### 2d — Categorical Field Distributions

Inspecting raw numeric codes before decoding to identify undocumented values.

In [10]:
raw_cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]

for col in raw_cat_cols:
    print(f"\n── {col} ──────────────────────")
    print(df[col].value_counts().sort_index().to_string())



── SEX ──────────────────────
SEX
1    11888
2    18112

── EDUCATION ──────────────────────
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51

── MARRIAGE ──────────────────────
MARRIAGE
0       54
1    13659
2    15964
3      323


---
## Step 3 — Rename Columns

All 25 columns are renamed to `snake_case` for PEP 8 compliance.  
An assertion confirms no column is silently skipped.

In [11]:
df = df.rename(columns=COLUMN_RENAME_MAP)

# Verify the rename was complete and correct
assert list(df.columns) == list(COLUMN_RENAME_MAP.values()), (
    "Rename produced unexpected column set."
)

print("Columns after rename:")
for i, col in enumerate(df.columns, start=1):
    print(f"  {i:>2}. {col}")


Columns after rename:
   1. customer_id
   2. credit_limit
   3. sex
   4. education
   5. marriage
   6. age
   7. pay_status_1
   8. pay_status_2
   9. pay_status_3
  10. pay_status_4
  11. pay_status_5
  12. pay_status_6
  13. bill_amount_1
  14. bill_amount_2
  15. bill_amount_3
  16. bill_amount_4
  17. bill_amount_5
  18. bill_amount_6
  19. pay_amount_1
  20. pay_amount_2
  21. pay_amount_3
  22. pay_amount_4
  23. pay_amount_5
  24. pay_amount_6
  25. default_next_month


---
## Step 4 — Decode Categorical Columns

Numeric codes for `sex`, `education`, and `marriage` are mapped to human-readable labels using `pd.Series.map()`.  

`.map()` is preferred over `.replace()` here because any **unmapped code becomes `NaN`** — a visible error — rather than silently surviving as a raw integer.

In [12]:
df['sex'] = df['sex'].map(SEX_MAP)
df['education'] = df['education'].map(EDUCATION_MAP)
df['marriage'] = df['marriage'].map(MARRIAGE_MAP)

# Assert no NaN was introduced (every code is covered by the maps)
for col in ['sex', 'education', 'marriage']:
    unmapped = df[col].isnull().sum()
    assert unmapped == 0, (
        f"{col} has {unmapped} unmapped code(s) — update the map."
    )

print("Decoded distributions:")
for col in ['sex', 'education', 'marriage']:
    print(f"\n── {col} ──────────────────────")
    print(df[col].value_counts().to_string())


Decoded distributions:

── sex ──────────────────────
sex
Female    18112
Male      11888

── education ──────────────────────
education
University         14030
Graduate School    10585
High School         4917
Unknown              345
Other                123

── marriage ──────────────────────
marriage
Single     15964
Married    13659
Other        323
Unknown       54


### 4b — `pay_status` Encoding Documentation

The six `pay_status_x` columns share the following encoding.  
This convention is adopted from the UCI supplementary notes and community consensus on the undocumented `-2` code:

| Value | Meaning |
|------:|:--------|
| `-2`  | No consumption that month (account inactive) — *undocumented in UCI codebook; treated as no delay* |
| `-1`  | Payment made in full (paid on time) |
| `0`   | Minimum payment paid / revolving credit in use |
| `1`   | Payment delayed by 1 month |
| `2–8` | Payment delayed by 2–8 months |

> **Risk signal:** only values **≥ 1** indicate delinquency and are used in feature engineering below.

In [13]:
print("PAY status value ranges:")
for col in PAY_STATUS_COLS:
    uniq = sorted(df[col].unique())
    print(
        f"  {col}: "
        f"min={df[col].min():>3}  "
        f"max={df[col].max():>3}  "
        f"unique={uniq}"
    )


PAY status value ranges:
  pay_status_1: min= -2  max=  8  unique=[np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
  pay_status_2: min= -2  max=  8  unique=[np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
  pay_status_3: min= -2  max=  8  unique=[np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
  pay_status_4: min= -2  max=  8  unique=[np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
  pay_status_5: min= -2  max=  8  unique=[np.int64(-2), np.int64(-1), np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
  pay_status_6: min= -2  max=  8  unique=[np.int64(-2

---
## Step 5 — Duplicates, Range Validation, and Edge-Case Flags

All cleaning decisions are stated and justified in line.

### 5a — Remove Duplicate Rows

In [14]:
initial_count = len(df)
df = df.drop_duplicates()
removed = initial_count - len(df)

print(f"Duplicate rows removed : {removed}")
print(f"Rows remaining         : {len(df):,}")


Duplicate rows removed : 0
Rows remaining         : 30,000


### 5b — Customer ID Uniqueness

In [15]:
dup_ids = df["customer_id"].duplicated().sum()
assert dup_ids == 0, (
    f"Found {dup_ids} duplicate customer IDs — investigate before proceeding."
)
print(f"Customer ID uniqueness : ✓  (0 duplicates)")


Customer ID uniqueness : ✓  (0 duplicates)


### 5c — Numeric Range Validation

Ranges checked against domain knowledge:

- **Age** — credit card eligibility requires ≥ 18; 100 used as an upper sanity bound.
- **Credit limit** — must be strictly positive.
- **Pay amounts** — cannot be negative (payments cannot go into reverse).

In [16]:
# ── Age ─────────────────────────────────────────────────────────────
invalid_age = df[(df['age'] < 18) | (df['age'] > 100)]
print(f"Age out of range [18, 100] : {len(invalid_age)} rows")

# ── Credit limit ─────────────────────────────────────────────────────
invalid_limit = df[df['credit_limit'] <= 0]
print(f"Non-positive credit limit  : {len(invalid_limit)} rows")

# ── Payment amounts ───────────────────────────────────────────────────
neg_pay_found = False
for col in PAY_AMOUNT_COLS:
    neg = (df[col] < 0).sum()
    if neg:
        print(f"  Negative {col} : {neg} rows")
        neg_pay_found = True
if not neg_pay_found:
    print("Pay amounts (all months)   : ✓  no negatives")

# ── Negative bill amounts (expected — credit balance from overpayment)
neg_bills = {}
for col in BILL_AMOUNT_COLS:
    n = (df[col] < 0).sum()
    if n:
        neg_bills[col] = n
print("\nNegative bill amounts (credit balances — not errors):")
for col, n in neg_bills.items():
    print(f"  {col}: {n:,} rows")


Age out of range [18, 100] : 0 rows
Non-positive credit limit  : 0 rows
Pay amounts (all months)   : ✓  no negatives

Negative bill amounts (credit balances — not errors):
  bill_amount_1: 590 rows
  bill_amount_2: 669 rows
  bill_amount_3: 655 rows
  bill_amount_4: 675 rows
  bill_amount_5: 655 rows
  bill_amount_6: 688 rows


### 5d — Edge-Case Flags

Three anomalous patterns identified during EDA are **flagged rather than dropped**. Dropping them would silently remove signal: for example, the 495 'ghost accounts' (Group A) carry a **39.6 % default rate** despite showing no bill activity.

| Flag | Rows | Root cause |
|:-----|-----:|:-----------|
| `flag_delay_no_bill` | 1 312 | `pay_status_1 = 1` but `bill_amount_1 = 0` — stale flag, timing lag, or fee-only delinquency |
| `flag_over_limit` | 3 931 | Bill exceeds credit limit — interest / fee accrual; strong risk signal |
| `flag_credit_balance` | 1 930 | Negative bill — customer overpaid; not a risk signal but guards ratio features |

In [17]:
# ── Flag 1: delay status with no outstanding bill ────────────────────
# pay_status_1 ≥ 1 indicates a recorded delay, yet bill_amount_1 = 0
# means no current outstanding balance. Three sub-causes identified:
#   A) Ghost accounts  — zero activity all 6 months (39.6 % default)
#   B) Settled after flag — bill cleared after status was stamped
#   C) Paid late — customer paid, balance cleared, flag not reset
df['flag_delay_no_bill'] = (
    (df['pay_status_1'] >= 1) & (df['bill_amount_1'] == 0)
).astype(int)

# ── Flag 2: over-limit account ───────────────────────────────────────
# True in any month where bill amount exceeds the granted credit limit.
# Caused by interest / penalty fees accruing beyond the ceiling.
df['flag_over_limit'] = (
    df[BILL_AMOUNT_COLS]
    .gt(df['credit_limit'], axis=0)
    .any(axis=1)
    .astype(int)
)

# ── Flag 3: credit balance ────────────────────────────────────────────
# True in any month where bill amount is negative (customer overpaid).
# Guards pay-to-bill ratio calculations from meaningless values.
df['flag_credit_balance'] = (
    df[BILL_AMOUNT_COLS].lt(0).any(axis=1).astype(int)
)

# ── Summary ──────────────────────────────────────────────────────────
flag_cols = [
    'flag_delay_no_bill',
    'flag_over_limit',
    'flag_credit_balance',
]
print("Edge-case flags added:")
for flag in flag_cols:
    n = df[flag].sum()
    pct = n / len(df) * 100
    print(f"  {flag:<25}: {n:>5,}  ({pct:.1f} %)")


Edge-case flags added:
  flag_delay_no_bill       : 1,312  (4.4 %)
  flag_over_limit          : 3,931  (13.1 %)
  flag_credit_balance      : 1,930  (6.4 %)


---
## Step 6 — Feature Engineering

Six behavioural features are engineered from the raw columns.  
Each is implemented as a **user-defined function (UDF)** with a docstring, enabling:

- Transparent logic that can be reviewed independently
- Reuse across phases and future iterations
- Easy unit-testing if needed

| Feature | Description |
|:--------|:------------|
| `avg_bill_amount` | Mean of 6 monthly bill amounts (NTD) |
| `avg_pay_amount` | Mean of 6 monthly payment amounts (NTD) |
| `pay_to_bill_ratio` | `avg_pay / avg_bill` — payment capacity indicator |
| `num_delayed_months` | Count of months with `pay_status ≥ 1` |
| `max_delay_months` | Worst single-month delay over 6 months |
| `balance_trend` | `bill_amount_1 − bill_amount_6` — rising or falling debt |

**Ratio edge case:** `pay_to_bill_ratio` is intentionally `NaN` when `avg_bill_amount <= 0`. In this dataset, 1,071 customers have a zero or negative average bill, usually because of no bill activity or a credit balance. Dividing by those values would create a misleading or undefined repayment measure, so these rows are retained and flagged rather than imputed or dropped.

### 6a — Feature Functions

In [18]:
def compute_avg_bill(row: pd.Series) -> float:
    """Return mean of the six monthly bill amounts.

    Negative values (credit balances) are included so the average
    reflects true net balance rather than raw spend volume.
    """
    return row[BILL_AMOUNT_COLS].mean()


def compute_avg_payment(row: pd.Series) -> float:
    """Return mean of the six monthly payment amounts."""
    return row[PAY_AMOUNT_COLS].mean()


def compute_pay_to_bill_ratio(row: pd.Series) -> float:
    """Return mean payment divided by mean bill amount.

    Returns NaN when the average bill is zero or negative to prevent
    misleading ratios — e.g. a small payment against a negative bill
    would produce a large positive number with no analytical meaning.
    A value of 1.0 means full repayment on average; below 1.0
    indicates under-payment relative to the statement balance.
    """
    avg_bill = compute_avg_bill(row)
    if avg_bill <= 0:
        return np.nan
    return compute_avg_payment(row) / avg_bill


def compute_num_delayed_months(row: pd.Series) -> int:
    """Return count of months with a recorded delay (pay_status ≥ 1)."""
    return int((row[PAY_STATUS_COLS] >= 1).sum())


def compute_max_delay(row: pd.Series) -> int:
    """Return the worst single-month delay over the 6-month window.

    Returns 0 when no delay is present (all statuses are ≤ 0).
    """
    delays = row[PAY_STATUS_COLS][row[PAY_STATUS_COLS] >= 1]
    return int(delays.max()) if not delays.empty else 0


def compute_balance_trend(row: pd.Series) -> float:
    """Return recent-balance trend: bill_amount_1 minus bill_amount_6.

    Positive → outstanding balance has grown (increasing exposure).
    Negative → balance has been reduced (improving financial health).
    Negative bill amounts are included to capture true net movement.
    """
    return row['bill_amount_1'] - row['bill_amount_6']


print("Feature functions defined ✓")


Feature functions defined ✓


### 6b — Apply Features and Validate

In [19]:
print("Engineering features — this may take a few seconds...")

df['avg_bill_amount'] = df.apply(compute_avg_bill, axis=1)
df['avg_pay_amount'] = df.apply(compute_avg_payment, axis=1)
df['pay_to_bill_ratio'] = df.apply(compute_pay_to_bill_ratio, axis=1)
df['num_delayed_months'] = df.apply(
    compute_num_delayed_months, axis=1
)
df['max_delay_months'] = df.apply(compute_max_delay, axis=1)
df['balance_trend'] = df.apply(compute_balance_trend, axis=1)

# ── Sanity checks ────────────────────────────────────────────────────
assert df['num_delayed_months'].between(0, 6).all(), (
    "num_delayed_months must be in [0, 6] — check UDF logic."
)
assert df['max_delay_months'].ge(0).all(), (
    "max_delay_months must be non-negative."
)

# ── Feature summary ──────────────────────────────────────────────────
engineered_cols = [
    'avg_bill_amount',
    'avg_pay_amount',
    'pay_to_bill_ratio',
    'num_delayed_months',
    'max_delay_months',
    'balance_trend',
]
print("\nEngineered feature statistics:")
df[engineered_cols].describe().round(2)


Engineering features — this may take a few seconds...

Engineered feature statistics:


,avg_bill_amount,avg_pay_amount,pay_to_bill_ratio,num_delayed_months,max_delay_months,balance_trend
count,"30,000.00","30,000.00","28,929.00","30,000.00","30,000.00","30,000.00"
mean,"44,976.95","5,275.23",0.49,0.83,0.68,"12,351.57"
std,"63,260.72","10,137.95",5.61,1.55,1.07,"43,922.42"
min,"-56,043.17",0.00,0.00,0.00,0.00,"-428,791.00"
25%,"4,781.33","1,113.29",0.04,0.00,0.00,"-2,963.00"
50%,"21,051.83","2,397.17",0.09,0.00,0.00,923.00
75%,"57,104.42","5,583.92",0.62,1.00,2.00,"19,793.75"
max,"877,313.83","627,344.33",797.00,6.00,8.00,"708,323.00"


**Interpretation - Feature Validation**

The engineered fields retain the full customer population while making repayment, delinquency, and balance movement comparable across six months. A missing `pay_to_bill_ratio` is an intentional statement that repayment capacity cannot be interpreted for a zero or negative average bill; it is not a missing-value cleaning failure. The edge-case flags preserve unusual records for later analysis instead of silently removing potential risk signal. These fields support historical monitoring and segmentation, not causal claims or automated decisions on their own.

---
## Step 7 — Persist Cleaned Dataset

The cleaned, enriched DataFrame is exported to CSV for use in Phases 2–5.  
Setting `index=False` omits the auto-generated pandas integer index.

The CSV deliberately remains the stable 34-column Phase 1 asset. Phase-specific reporting fields are regenerated from these saved columns rather than persisted: Phase 2 creates `age_group` with fixed documented bins and `credit_limit_tier` with its documented portfolio `qcut`; Phases 2–4 create `credit_utilization` as `avg_bill_amount / credit_limit`, clipped at zero. This avoids freezing a sample-dependent tier in the shared CSV while keeping every later phase standalone and reproducible.

In [20]:
df.to_csv(OUTPUT_PATH, index=False)

print(f"Cleaned dataset saved → {OUTPUT_PATH}")
print(f"  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]}")

# ── Final column inventory ────────────────────────────────────────────
original_cols = list(COLUMN_RENAME_MAP.values())
flag_cols_added = [
    'flag_delay_no_bill',
    'flag_over_limit',
    'flag_credit_balance',
]
feat_cols_added = [
    'avg_bill_amount',
    'avg_pay_amount',
    'pay_to_bill_ratio',
    'num_delayed_months',
    'max_delay_months',
    'balance_trend',
]
print("\n── Original columns (renamed) ──────────────────────────")
for col in original_cols:
    print(f"  {col}")
print("\n── Added: edge-case flags ───────────────────────────────")
for col in flag_cols_added:
    print(f"  {col}")
print("\n── Added: engineered features ──────────────────────────")
for col in feat_cols_added:
    print(f"  {col}")


Cleaned dataset saved → cleaned_credit_data.csv
  Rows    : 30,000
  Columns : 34

── Original columns (renamed) ──────────────────────────
  customer_id
  credit_limit
  sex
  education
  marriage
  age
  pay_status_1
  pay_status_2
  pay_status_3
  pay_status_4
  pay_status_5
  pay_status_6
  bill_amount_1
  bill_amount_2
  bill_amount_3
  bill_amount_4
  bill_amount_5
  bill_amount_6
  pay_amount_1
  pay_amount_2
  pay_amount_3
  pay_amount_4
  pay_amount_5
  pay_amount_6
  default_next_month

── Added: edge-case flags ───────────────────────────────
  flag_delay_no_bill
  flag_over_limit
  flag_credit_balance

── Added: engineered features ──────────────────────────
  avg_bill_amount
  avg_pay_amount
  pay_to_bill_ratio
  num_delayed_months
  max_delay_months
  balance_trend


---
## Phase 1 Summary

| Step | Action | Outcome |
|-----:|:-------|:--------|
| 1 | Loaded CSV with `header=1` (double-header fix) | 30 000 rows × 25 cols confirmed |
| 2 | Initial audit | 0 nulls, 0 duplicates, categorical codes inspected |
| 3 | Renamed 25 columns to `snake_case` | PEP 8 compliant names throughout |
| 4 | Decoded `sex`, `education`, `marriage` | Undocumented codes → `Unknown` (345 + 54 rows retained) |
| 4b | Documented `pay_status` encoding | `-2` treated as no delay; `≥ 1` = delinquency |
| 5a | Removed duplicate rows | 0 removed |
| 5b | Validated ID uniqueness | 0 duplicate IDs |
| 5c | Validated numeric ranges | Age, credit limit, pay amounts all valid |
| 5d | Added 3 edge-case flags | `flag_delay_no_bill`, `flag_over_limit`, `flag_credit_balance` |
| 6 | Engineered 6 behavioural features | `avg_bill`, `avg_pay`, `pay_to_bill_ratio`, `num_delayed_months`, `max_delay_months`, `balance_trend` |
| 7 | Exported `cleaned_credit_data.csv` | 30 000 rows × 34 columns |

**Proceed to Phase 2 — Exploratory Data Analysis.**